# Aula 05 — Evolução dos Modelos de Representação e Persistência

**IFPB PPGTI — Banco de Dados**

Este notebook reúne os dois demos práticos da aula:

| Demo | Tema | Conceito central |
|------|------|------------------|
| 1 | JSONB | o NoSQL dentro do relacional |
| 2 | pgvector | banco vetorial integrado |

> **Para estudo adicional:** o arquivo `demo_hybrid_search.py` e o script
> `sql/04_hybrid_search.sql` demonstram busca híbrida com Reciprocal Rank Fusion (RRF).

**Pré-requisito:** Docker rodando (`docker compose up -d`).


## Setup — conexão e utilitários

In [5]:
!pip install psycopg2
!pip install psycopg2-binary


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [32]:
import json, time, textwrap
from pathlib import Path
import psycopg2, psycopg2.extras
import numpy as np

# ── Conexão ──────────────────────────────────────────────────
conn = psycopg2.connect(
    host="localhost", port=5433,
    dbname="aula05", user="aula05", password="aula05"
)
print("Conectado ao PostgreSQL:", conn.get_dsn_parameters()['dbname'])

# ── Helper de exibição ────────────────────────────────────────
def show(rows, headers=None):
    """Imprime linhas de cursor como tabela simples."""
    if not rows:
        print("  (sem resultados)")
        return
    if headers:
        print("  " + " | ".join(f"{h:20s}" for h in headers))
        print("  " + "-" * (23 * len(headers)))
    for row in rows:
        print("  " + " | ".join(f"{str(v)[:20]:20s}" for v in row))

Conectado ao PostgreSQL: aula05


## Schema — extensões, tabelas e índices

Cria tudo que os demos precisam: extensões `vector` e `pg_trgm`,
tabelas `eventos` e `artigos`, índices GIN e HNSW.

Execute esta célula **uma vez** após subir o Docker (`docker compose up -d`).

In [33]:
DDL = """
-- ── Extensões ──────────────────────────────────────────────────
CREATE EXTENSION IF NOT EXISTS vector;    -- pgvector
CREATE EXTENSION IF NOT EXISTS pg_trgm;   -- trigram para FTS

-- ── Demo 1: JSONB ───────────────────────────────────────────────
DROP TABLE IF EXISTS eventos CASCADE;

CREATE TABLE eventos (
    id         SERIAL PRIMARY KEY,
    tipo       VARCHAR(50)  NOT NULL,
    payload    JSONB        NOT NULL,
    created_at TIMESTAMPTZ  DEFAULT now()
);

-- GIN index: habilita @>, ?, ?& em O(log n)
CREATE INDEX idx_eventos_payload ON eventos USING GIN (payload);
CREATE INDEX idx_eventos_tipo    ON eventos (tipo);

-- ── Demo 2 & 3: pgvector + FTS ──────────────────────────────────
DROP TABLE IF EXISTS artigos CASCADE;

CREATE TABLE artigos (
    id        SERIAL PRIMARY KEY,
    titulo    TEXT NOT NULL,
    area      TEXT NOT NULL,
    resumo    TEXT NOT NULL,
    ano       INT,
    -- vetor 384 dims — modelo all-MiniLM-L6-v2
    embedding vector(384),
    -- coluna computed STORED — atualizada automaticamente
    tsv       TSVECTOR GENERATED ALWAYS AS (
                  to_tsvector('portuguese', titulo || ' ' || resumo)
              ) STORED
);

-- Índice HNSW: m=16 conexões por nó, ef_construction=64
CREATE INDEX idx_artigos_hnsw ON artigos
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);

-- Índice GIN para Full-Text Search
CREATE INDEX idx_artigos_tsv ON artigos USING GIN (tsv);
"""

with conn.cursor() as cur:
    cur.execute(DDL)
conn.commit()

print("Schema criado com sucesso.")
print("  extensões : vector, pg_trgm")
print("  tabelas   : eventos, artigos")
print("  índices   : GIN (payload), GIN (tsv), HNSW (embedding)")

Schema criado com sucesso.
  extensões : vector, pg_trgm
  tabelas   : eventos, artigos
  índices   : GIN (payload), GIN (tsv), HNSW (embedding)


---
## Demo 1 — JSONB: o NoSQL dentro do relacional

> **Tese:** o PostgreSQL com JSONB funciona como banco de documentos,
> com a vantagem de poder fazer JOIN com tabelas relacionais na mesma query.
> Nenhum MongoDB faz isso.
>
> Referência: Stonebraker, M. (2015). 'The land sharks are on the squawk box.' *ACM Queue.*

### 1.1 Inserir eventos com schemas heterogêneos

In [34]:
EVENTOS = [
    {"tipo": "sensor_temp",
     "payload": {"device": "s01", "location": "sala-A", "temp": 28.5, "unit": "C"}},
    {"tipo": "sensor_temp",
     "payload": {"device": "s02", "location": "sala-B", "temp": 21.0, "unit": "C"}},
    {"tipo": "sensor_temp",
     "payload": {"device": "s03", "location": "CPD",    "temp": 18.3, "unit": "C"}},
    {"tipo": "user_action",
     "payload": {"user_id": 42, "action": "click", "page": "/home",
                 "metadata": {"browser": "firefox", "os": "linux"}}},
    {"tipo": "user_action",
     "payload": {"user_id": 99, "action": "purchase", "page": "/checkout",
                 "items": ["SKU-001", "SKU-042"], "total": 199.90}},
    {"tipo": "pedido",
     "payload": {"order_id": "ORD-001",
                 "customer": {"name": "Maria Silva", "tier": "gold"},
                 "items": [{"sku": "SKU-001", "qty": 2, "price": 49.90},
                            {"sku": "SKU-042", "qty": 1, "price": 100.10}],
                 "total": 199.90, "payment": "pix"}},
    {"tipo": "pedido",
     "payload": {"order_id": "ORD-002",
                 "customer": {"name": "João Ferreira", "tier": "silver"},
                 "items": [{"sku": "SKU-007", "qty": 3, "price": 15.00}],
                 "total": 45.00, "payment": "credito"}},
]

with conn.cursor() as cur:
    cur.execute("TRUNCATE eventos RESTART IDENTITY")
    for ev in EVENTOS:
        cur.execute(
            "INSERT INTO eventos (tipo, payload) VALUES (%s, %s)",
            (ev["tipo"], json.dumps(ev["payload"]))
        )
conn.commit()
print(f"{len(EVENTOS)} eventos inseridos.")
print("Tipos presentes:", {e['tipo'] for e in EVENTOS})

7 eventos inseridos.
Tipos presentes: {'sensor_temp', 'user_action', 'pedido'}


### 1.2 Operadores JSONB: `->`, `->>`, `@>`, `#>>`

In [8]:
# Operador ->> (extrai como texto) — sensores com temp > 20°C
with conn.cursor() as cur:
    cur.execute("""
        SELECT payload->>'device' AS device,
               payload->>'location' AS location,
               (payload->>'temp')::float AS temp_c
        FROM   eventos
        WHERE  tipo = 'sensor_temp'
          AND  (payload->>'temp')::float > 20
        ORDER  BY temp_c DESC
    """)
    rows = cur.fetchall()

print("Sensores > 20°C (operador ->>):")
show(rows, ["device", "location", "temp_c"])

Sensores > 20°C (operador ->>):
  device               | location             | temp_c              
  ---------------------------------------------------------------------
  s01                  | sala-A               | 28.5                
  s02                  | sala-B               | 21.0                


In [9]:
# Operador @> (containment) — busca campo aninhado
with conn.cursor() as cur:
    cur.execute("""
        SELECT id, payload->>'user_id' AS user_id, payload->>'page' AS page
        FROM   eventos
        WHERE  tipo = 'user_action'
          AND  payload @> '{"metadata": {"browser": "firefox"}}'
    """)
    rows = cur.fetchall()

print("Usuários com browser=firefox (operador @>):")
show(rows, ["id", "user_id", "page"])

Usuários com browser=firefox (operador @>):
  id                   | user_id              | page                
  ---------------------------------------------------------------------
  4                    | 42                   | /home               


In [10]:
# Operador #>> — caminho aninhado como array de strings
with conn.cursor() as cur:
    cur.execute("""
        SELECT payload #>> '{customer,name}' AS nome,
               payload #>> '{customer,tier}' AS tier,
               (payload->>'total')::numeric   AS total
        FROM   eventos
        WHERE  tipo = 'pedido'
        ORDER  BY total DESC
    """)
    rows = cur.fetchall()

print("Clientes de pedidos (operador #>>):")
show(rows, ["nome", "tier", "total"])

Clientes de pedidos (operador #>>):
  nome                 | tier                 | total               
  ---------------------------------------------------------------------
  Maria Silva          | gold                 | 199.9               
  João Ferreira        | silver               | 45.0                


### 1.3 `jsonb_array_elements` — iterar arrays dentro do JSON

In [11]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT e.id                          AS pedido_id,
               item->>'sku'                  AS sku,
               (item->>'qty')::int           AS qty,
               (item->>'price')::numeric     AS preco
        FROM   eventos e,
               jsonb_array_elements(e.payload->'items') AS item
        WHERE  e.tipo = 'pedido'
        ORDER  BY e.id, sku
    """)
    rows = cur.fetchall()

print("Itens de pedidos (jsonb_array_elements):")
show(rows, ["pedido_id", "sku", "qty", "preco"])

Itens de pedidos (jsonb_array_elements):
  pedido_id            | sku                  | qty                  | preco               
  --------------------------------------------------------------------------------------------
  6                    | SKU-001              | 2                    | 49.9                
  6                    | SKU-042              | 1                    | 100.1               
  7                    | SKU-007              | 3                    | 15.0                


### 1.4 EXPLAIN — comportamento do planner com tabelas pequenas

In [35]:
# Com poucos registros o planner escolhe Seq Scan (mais barato que acessar o índice).
# Para demonstrar que o índice GIN existe e é usado em escala, forçamos com enable_seqscan=off.

with conn.cursor() as cur:

    # ── Plano real (sem forçar) ──────────────────────────────
    cur.execute("""
        EXPLAIN (FORMAT TEXT, COSTS OFF)
        SELECT * FROM eventos
        WHERE payload @> '{"metadata": {"browser": "firefox"}}'
    """)
    plan_real = cur.fetchall()

    # ── Plano forçando índice (simula tabela grande) ──────────
    cur.execute("SET enable_seqscan = off")
    cur.execute("""
        EXPLAIN (FORMAT TEXT, COSTS OFF)
        SELECT * FROM eventos
        WHERE payload @> '{"metadata": {"browser": "firefox"}}'
    """)
    plan_gin = cur.fetchall()
    cur.execute("SET enable_seqscan = on")  # restaura o comportamento padrão

print("Plano real (tabela pequena — Seq Scan é mais barato):")
for line in plan_real:
    print(" ", line[0])

print("\nPlano com índice forçado (comportamento em tabelas grandes):")
for line in plan_gin:
    print(" ", line[0])

print("\nNota: o GIN (Generalized Inverted Index) é usado automaticamente quando a tabela tem muitas linhas.")


Plano real (tabela pequena — Seq Scan é mais barato):
  Seq Scan on eventos
    Filter: (payload @> '{"metadata": {"browser": "firefox"}}'::jsonb)

Plano com índice forçado (comportamento em tabelas grandes):
  Bitmap Heap Scan on eventos
    Recheck Cond: (payload @> '{"metadata": {"browser": "firefox"}}'::jsonb)
    ->  Bitmap Index Scan on idx_eventos_payload
          Index Cond: (payload @> '{"metadata": {"browser": "firefox"}}'::jsonb)

Nota: o GIN (Generalized Inverted Index) é usado automaticamente quando a tabela tem muitas linhas.


### 1.5 Agregação sobre campo JSONB

In [16]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT payload->>'payment'                   AS metodo,
               COUNT(*)                              AS qtd,
               SUM((payload->>'total')::numeric)     AS receita
        FROM   eventos
        WHERE  tipo = 'pedido'
        GROUP  BY payload->>'payment'
        ORDER  BY receita DESC
    """)
    rows = cur.fetchall()

print("Receita por método de pagamento:")
show(rows, ["metodo", "qtd", "receita"])

Receita por método de pagamento:
  metodo               | qtd                  | receita             
  ---------------------------------------------------------------------
  pix                  | 1                    | 199.9               
  credito              | 1                    | 45.0                


---
## Demo 2 — pgvector: banco vetorial integrado

> **Tese:** não é preciso manter Pinecone ou Qdrant para a maioria dos cenários de RAG.
> pgvector permite combinar busca vetorial + filtros SQL + JOINs na mesma query.
>
> Referência: Malkov & Yashunin (2020). *Efficient and Robust ANN Search Using HNSW.* IEEE TPAMI.

### 2.1 Carregar modelo e gerar embeddings

In [18]:
!pip install sentence_transformers


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: 384 dims, ~80MB, ótimo para demonstração
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Modelo carregado. Dimensão de saída: {model.get_sentence_embedding_dimension()}")

/Users/diegopessoa/projects/ifpb/MPTI-BD-2026.1/exemplos/aula-04-streaming-varejo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12806.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado. Dimensão de saída: 384


/var/folders/gl/9h1dn5w11b7ffj11pym6g7yr0000gn/T/ipykernel_84599/3358492317.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Modelo carregado. Dimensão de saída: {model.get_sentence_embedding_dimension()}")


In [36]:
ARTIGOS = [
    {"titulo": "Spanner: Google's Globally-Distributed Database",
     "area": "newsql", "ano": 2012,
     "resumo": "Spanner usa TrueTime com clocks atômicos para consistência externa. Suporta transações ACID distribuídas com SQL completo."},
    {"titulo": "Dynamo: Amazon's Highly Available Key-value Store",
     "area": "nosql", "ano": 2007,
     "resumo": "Dynamo usa consistent hashing e eventual consistency. Inspirou Cassandra e o movimento NoSQL."},
    {"titulo": "Bigtable: A Distributed Storage System for Structured Data",
     "area": "nosql", "ano": 2006,
     "resumo": "Bigtable é um modelo de dados esparso e distribuído do Google. Influenciou HBase e Cassandra."},
    {"titulo": "What's Really New with NewSQL?",
     "area": "newsql", "ano": 2016,
     "resumo": "Análise crítica do NewSQL: SQL + ACID + escalabilidade horizontal. CockroachDB, TiDB e VoltDB."},
    {"titulo": "DuckDB: an Embeddable Analytical Database",
     "area": "olap", "ano": 2019,
     "resumo": "DuckDB executa queries analíticas sobre Parquet e CSV. Alternativa ao SQLite para analytics."},
    {"titulo": "pgvector: Open-Source Vector Similarity Search for Postgres",
     "area": "vetorial", "ano": 2021,
     "resumo": "pgvector adiciona índices HNSW e IVFFlat ao PostgreSQL. Permite combinar vetorial com SQL relacional."},
    {"titulo": "Lakehouse: A New Generation of Open Platforms",
     "area": "arquitetura", "ano": 2021,
     "resumo": "Lakehouse une data lake e warehouse. Delta Lake e Iceberg implementam ACID sobre object storage."},
    {"titulo": "Efficient and Robust ANN Search Using HNSW",
     "area": "vetorial", "ano": 2020,
     "resumo": "HNSW é baseado em grafos hierárquicos navegáveis. Melhor recall/latência que IVFFlat. Base do pgvector, Qdrant e FAISS."},
    {"titulo": "CockroachDB: The Resilient Geo-Distributed SQL Database",
     "area": "newsql", "ano": 2020,
     "resumo": "CockroachDB usa Raft consensus por range. Isolamento serializável. Sobrevive a falhas de datacenter."},
    {"titulo": "Hybrid Search Combining BM25 and Dense Retrieval",
     "area": "busca", "ano": 2021,
     "resumo": "Busca híbrida combina BM25 lexical com embeddings densos. RRF agrega rankings. Supera cada método isolado."},
    {"titulo": "Retrieval-Augmented Generation for Knowledge-Intensive NLP",
     "area": "ia", "ano": 2020,
     "resumo": "RAG combina recuperação densa com geração por LLMs. Reduz alucinações. Usa embeddings para busca semântica."},
    {"titulo": "MongoDB: Document Database for Modern Applications",
     "area": "nosql", "ano": 2019,
     "resumo": "MongoDB armazena documentos BSON. A partir v4.0 suporta ACID multi-documento. Converge para modelo relacional."},
]

textos = [f"{a['titulo']} {a['resumo']}" for a in ARTIGOS]
embeddings = model.encode(textos, normalize_embeddings=True)
print(f"{len(embeddings)} embeddings gerados, shape={embeddings.shape}")

12 embeddings gerados, shape=(12, 384)


### 2.2 Inserir vetores no PostgreSQL

In [37]:
with conn.cursor() as cur:
    cur.execute("TRUNCATE artigos RESTART IDENTITY")
    for artigo, emb in zip(ARTIGOS, embeddings):
        cur.execute(
            "INSERT INTO artigos (titulo, area, resumo, ano, embedding) VALUES (%s,%s,%s,%s,%s)",
            (artigo["titulo"], artigo["area"], artigo["resumo"], artigo["ano"], emb.tolist())
        )
conn.commit()
print(f"{len(ARTIGOS)} artigos inseridos com embeddings de 384 dims.")

12 artigos inseridos com embeddings de 384 dims.


### 2.3 Busca por similaridade semântica (top-k)

In [40]:
def busca_semantica(query: str, area: str = None, k: int = 5):
    """Busca top-k artigos por similaridade de cosseno."""
    q_emb = model.encode([query], normalize_embeddings=True)[0].tolist()
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        if area:
            cur.execute("""
                SELECT titulo, area, ano,
                       1 - (embedding <=> %s::vector) AS similaridade
                FROM   artigos
                WHERE  area = %s
                ORDER  BY embedding <=> %s::vector
                LIMIT  %s
            """, (q_emb, area, q_emb, k))
        else:
            cur.execute("""
                SELECT titulo, area, ano,
                       1 - (embedding <=> %s::vector) AS similaridade
                FROM   artigos
                ORDER  BY embedding <=> %s::vector
                LIMIT  %s
            """, (q_emb, q_emb, k))
        return cur.fetchall()

# ── Teste 1: conceito geral ───────────────────────────────────
query = "Geo distribuídas para bancos newsql"
print(f"Query: '{query}'")
results = busca_semantica(query)
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['area']}] {r['titulo']} ({r['ano']}) — sim={float(r['similaridade'])*100:.1f}%")

Query: 'Geo distribuídas para bancos newsql'
  1. [newsql] CockroachDB: The Resilient Geo-Distributed SQL Database (2020) — sim=48.6%
  2. [newsql] What's Really New with NewSQL? (2016) — sim=45.3%
  3. [newsql] Spanner: Google's Globally-Distributed Database (2012) — sim=39.3%
  4. [olap] DuckDB: an Embeddable Analytical Database (2019) — sim=33.0%
  5. [nosql] MongoDB: Document Database for Modern Applications (2019) — sim=32.5%


In [41]:
# ── Teste 2: busca vetorial + filtro SQL (área específica) ────
query = "busca semântica com embeddings e índices aproximados"
print(f"Query: '{query}' [filtro: area='vetorial']")
results = busca_semantica(query, area="vetorial")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['titulo']} ({r['ano']}) — sim={float(r['similaridade'])*100:.1f}%")

Query: 'busca semântica com embeddings e índices aproximados' [filtro: area='vetorial']
  1. pgvector: Open-Source Vector Similarity Search for Postgres (2021) — sim=35.9%
  2. Efficient and Robust ANN Search Using HNSW (2020) — sim=31.2%


### 2.4 Comparativo de operadores de distância

In [42]:
# Comparar <=> (cosine), <-> (L2), <#> (inner product)
# Para vetores normalizados, cosine e inner product são equivalentes
with conn.cursor() as cur:
    cur.execute("""
        SELECT a.titulo,
               round((a.embedding <=>  b.embedding)::numeric, 4) AS dist_cosine,
               round((a.embedding <->  b.embedding)::numeric, 4) AS dist_l2,
               round((a.embedding <#>  b.embedding)::numeric, 4) AS neg_inner_prod
        FROM   artigos a
        CROSS  JOIN artigos b
        WHERE  b.titulo LIKE '%Spanner%'
          AND  a.titulo != b.titulo
        ORDER  BY dist_cosine
        LIMIT  5
    """)
    rows = cur.fetchall()

print("Top-5 mais próximos de 'Spanner' por diferentes métricas:")
show(rows, ["titulo", "cosine", "L2", "inner_prod"])

Top-5 mais próximos de 'Spanner' por diferentes métricas:
  titulo               | cosine               | L2                   | inner_prod          
  --------------------------------------------------------------------------------------------
  CockroachDB: The Res | 0.4184               | 0.9147               | -0.5816             
  Bigtable: A Distribu | 0.4809               | 0.9807               | -0.5191             
  DuckDB: an Embeddabl | 0.5491               | 1.0479               | -0.4509             
  Dynamo: Amazon's Hig | 0.5837               | 1.0804               | -0.4163             
  MongoDB: Document Da | 0.6107               | 1.1051               | -0.3893             


### 2.5 Comparativo: HNSW vs busca exata (latência)

In [43]:
q_emb = model.encode(["inteligência artificial e recuperação de informação"], normalize_embeddings=True)[0].tolist()

with conn.cursor() as cur:
    # Seq scan (exato)
    cur.execute("SET enable_indexscan=off; SET enable_bitmapscan=off;")
    t0 = time.perf_counter()
    cur.execute("SELECT titulo FROM artigos ORDER BY embedding <=> %s::vector LIMIT 3", (q_emb,))
    cur.fetchall()
    exact_ms = (time.perf_counter() - t0) * 1000

    # HNSW
    cur.execute("SET enable_indexscan=on; SET enable_bitmapscan=on;")
    t0 = time.perf_counter()
    cur.execute("SELECT titulo FROM artigos ORDER BY embedding <=> %s::vector LIMIT 3", (q_emb,))
    cur.fetchall()
    hnsw_ms = (time.perf_counter() - t0) * 1000
    conn.commit()

print(f"Corpus: {len(ARTIGOS)} artigos (corpus pequeno — diferença cresce com escala)")
print(f"  Seq scan (exato) : {exact_ms:.3f} ms")
print(f"  HNSW (aprox.)    : {hnsw_ms:.3f} ms")
print("")
print("Em corpora com 100k+ vetores o HNSW ganha de 10–100x.")
print("Malkov & Yashunin (2020): HNSW atinge 99% recall com latência sub-milissegundo.")

Corpus: 12 artigos (corpus pequeno — diferença cresce com escala)
  Seq scan (exato) : 5.504 ms
  HNSW (aprox.)    : 1.671 ms

Em corpora com 100k+ vetores o HNSW ganha de 10–100x.
Malkov & Yashunin (2020): HNSW atinge 99% recall com latência sub-milissegundo.


---
## Síntese: PostgreSQL como plataforma multimodelo

| Paradigma | Mecanismo | Operação chave |
|-----------|-----------|----------------|
| Relacional | tabelas + FK | `JOIN`, `GROUP BY` |
| Documento | `JSONB` + GIN | `@>`, `->`, `#>>` |
| Vetorial | `pgvector` + HNSW | `<=>`, `<->` |
| Full-Text | `tsvector` + GIN | `@@`, `ts_rank()` |
| Geoespacial | PostGIS | `ST_Distance()` |
| Séries temp. | TimescaleDB | `time_bucket()` |

**Questão de pesquisa aberta:** quando a especialização (Qdrant, Pinecone, MongoDB dedicado)
justifica o custo operacional adicional de manter múltiplos bancos?

> **Próximo passo:** busca híbrida — combinar `tsvector` (FTS) com `pgvector` usando
> Reciprocal Rank Fusion. Ver `demo_hybrid_search.py` para implementação completa.


In [ ]:
conn.close()
print("Conexão encerrada.")